# 02 — Transcripción en GPU

**Qué hace.** Transcribe los 10 audios con faster-whisper en una GPU T4 de Colab y mide lo que pide la Fase 2 de Task 1:
- dos tamaños de modelo;
- float16 contra int8;
- CPU contra GPU;
- VRAM pico;
- el efecto del glosario, con un intervalo pareado por clip.

Además mide la duración real de cada audio y prueba el motor completo con dos clips.

**Qué lee.** Los 10 audios, desde una carpeta privada de Drive, ubicados por su nombre original (`config.yaml → audios`). También lee las referencias `data/referencias/*.txt` y `config.yaml → asr`.

**Qué produce.** En Drive, en `HW_05_salidas/` (`config.yaml → colab.drive_salidas`), que replica `data/`:
- `cache/transcripciones/<corrida>/<clip>.json`: una transcripción por clip y corrida, con texto, palabras con probabilidad, modelo, precisión, device, tiempos y VRAM;
- `resultados/`: duraciones, mediciones por corrida, WER por clip y glosario.

Usa `src.audios`, `src.transcripcion`, `src.metricas` y `src.engine`; no reimplementa nada.

## Instrucciones para ejecutarlo

1. **Abrirlo en Colab desde GitHub.** Entra en este enlace: <https://colab.research.google.com/github/mmlis02/HW_05_202602/blob/feature/transcripcion/notebooks/02_transcripcion_gpu.ipynb>. Otra forma: en Colab, *Archivo → Abrir notebook → GitHub*, busca `mmlis02/HW_05_202602`, elige la rama `feature/transcripcion` y abre `notebooks/02_transcripcion_gpu.ipynb`.
2. **Elegir la GPU T4.** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: T4 GPU → Guardar.*
3. **Ejecutar todo.** *Entorno de ejecución → Ejecutar todas.*
   - Colab pedirá permiso para acceder a Drive: acéptalo con la cuenta que tiene los audios.
   - En la sección 1, **comprueba que la carpeta encontrada es la de tus audios**.
4. **Si la sesión se corta**, vuelve a *Ejecutar todas*. Cada transcripción se guarda en Drive en cuanto termina, y las que ya están se leen desde allí en lugar de repetirse.
5. **Al terminar, descarga dos cosas:**
   - **Este notebook con sus salidas:** *Archivo → Descargar → Descargar .ipynb*. Reemplaza `notebooks/02_transcripcion_gpu.ipynb`. Las salidas son la evidencia de las mediciones.
   - **La carpeta `HW_05_salidas` de tu Drive**, entera (en drive.google.com: clic derecho → *Descargar*). Su `cache/` va a `data/cache/` y su `resultados/` a `data/resultados/`.

   Los JSON no contienen rutas de Drive, solo el nombre de cada archivo de audio. Los audios no se descargan ni se suben al repo.

Duración aproximada con la T4: entre 45 y 90 minutos. La parte más lenta es la corrida en CPU (sección 7).

## 0. Preparar Colab

Clona el repositorio en la rama de este trabajo. La URL y la rama son lo único escrito a mano: hacen falta antes de que exista `config.yaml` en la máquina. Cuando el PR esté fusionado, `RAMA` pasa a ser `"main"`.

In [ ]:
import os

RAMA = "feature/transcripcion"
if not os.path.isdir("HW_05_202602"):
    !git clone --quiet --branch {RAMA} https://github.com/mmlis02/HW_05_202602.git
%cd HW_05_202602
!git checkout --quiet {RAMA} && git pull --quiet --ff-only && git log --oneline -1

### Versiones fijadas contra lo que trae Colab

`requirements.txt` fija las versiones del proyecto (por ejemplo, `pandas==3.1.0`). `requirements-gpu.txt` fija cuBLAS 12 y cuDNN 9 para la GPU. Colab trae sus propias versiones, y algunas las fija su paquete `google-colab`. Instalar `requirements.txt` completo en Colab reemplazaría esas versiones (pandas, entre otras) y puede romper el entorno.

La celda siguiente compara cada versión fijada con la instalada en Colab.

In [ ]:
import re
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd
from src.config import cargar_config

cfg_inicial = cargar_config()
colab = cfg_inicial["colab"]


def leer_fijadas(ruta):
    return dict(re.findall(r"^([A-Za-z0-9_.\-]+)==(\S+)", Path(ruta).read_text(), flags=re.M))


fijadas_gpu = leer_fijadas(cfg_inicial["rutas"]["requisitos_gpu"])
fijadas = {**leer_fijadas(cfg_inicial["rutas"]["requisitos"]), **fijadas_gpu}
a_instalar = [*colab["paquetes"], *fijadas_gpu]


def instalada(paquete):
    try:
        return version(paquete)
    except PackageNotFoundError:
        return None


versiones = pd.DataFrame([{"paquete": p, "fijada": v, "en_colab": instalada(p)} for p, v in fijadas.items()])
versiones["estado"] = [("igual" if c == f else "distinta" if c else "no instalada")
                       for f, c in zip(versiones.fijada, versiones.en_colab)]
versiones["se_instala_aqui"] = versiones.paquete.isin(a_instalar)
versiones

**Decisión.** En Colab se instalan, con la versión exacta de `requirements.txt`, solo los paquetes que este notebook necesita y que pueden cambiar el resultado (`config.yaml → colab.paquetes`):
- `faster-whisper` y `av`: la 19 de PyAV rompe faster-whisper 1.2.1;
- `jiwer` (WER y CER);
- `text2num` (normalización de números);
- `nvidia-ml-py` (VRAM);
- `nvidia-cublas-cu12` y `nvidia-cudnn-cu12` (cuDNN 9), de `requirements-gpu.txt`.

**Por qué las librerías de CUDA 12.** Colab pasó a Python 3.13 con una CUDA más nueva, y ya no trae `libcublas.so.12`. CTranslate2 4.x (el motor de faster-whisper) necesita cuBLAS para CUDA 12 y cuDNN 9, y sin ellas transcribir en GPU falla con `RuntimeError: Library libcublas.so.12 is not found or cannot be loaded`. Detectar la GPU no basta: eso funciona sin esas librerías. faster-whisper 1.2.1 indica instalarlas con pip. Las versiones están fijadas: cuBLAS 12.9 es la última serie de CUDA 12, y el driver de Colab la soporta.

pandas, PyYAML y lo demás se usan tal como vienen en Colab. El código del proyecto solo usa partes de pandas que son iguales en 2.x y 3.x. Streamlit, Jupyter y python-dotenv no hacen falta aquí.

Así, las versiones que afectan las mediciones quedan fijas, y el entorno de Colab queda intacto. Si pip pide reiniciar la sesión, reinicia (*Entorno de ejecución → Reiniciar sesión*) y vuelve a *Ejecutar todas*.

In [ ]:
instalar = " ".join(f"{p}=={fijadas[p]}" for p in a_instalar)
print("pip install", instalar)
!pip install --quiet {instalar}

In [ ]:
print(pd.DataFrame([{"paquete": p, "fijada": fijadas[p], "instalada": instalada(p)} for p in a_instalar]))
assert all(instalada(p) == fijadas[p] for p in a_instalar), "alguna versión no quedó como la fijada"
!pip check 2>&1 | grep -i -E "faster|ctranslate|^av |jiwer|text2num|nvidia" || echo "Sin conflictos en los paquetes instalados aquí."


### Que el proceso encuentre cuBLAS 12 y cuDNN 9

pip deja las librerías en `site-packages/nvidia/cublas/lib` y `site-packages/nvidia/cudnn/lib`, pero el cargador del sistema no las busca ahí. La documentación de faster-whisper propone `LD_LIBRARY_PATH`, y advierte que hay que definirla **antes de lanzar Python**. En Colab eso no es posible: cambiarla con el kernel ya iniciado no tiene efecto, porque el cargador la lee al arrancar, y no sobrevive a un reinicio de la sesión, porque el kernel vuelve a arrancar con el entorno de Colab.

**Solución elegida: precargarlas con `ctypes`.** `src.transcripcion.precargar_cuda12()` abre cada librería con `RTLD_GLOBAL`, en orden: `libcublasLt` antes que `libcublas` y las subbibliotecas de cuDNN antes que `libcudnn`, y reintenta si un orden no alcanza. Cuando CTranslate2 pide `libcublas.so.12` por nombre, el cargador encuentra la que ya está en el proceso. Se eligió esta opción sobre la de reiniciar porque no depende de un paso manual, funciona en el mismo proceso y se comprueba enseguida (celda de abajo). Tiene una condición: debe correr antes del primer uso de la GPU. Por eso va aquí, y `cargar_modelo` también la llama cada vez que el device es `cuda`.

Estos wheels no traen `__init__.py`, así que el atajo `nvidia.cublas.lib.__file__` de la documentación no funciona con ellos. La función busca las carpetas en `sys.path`.

In [ ]:
from src.transcripcion import precargar_cuda12

print("Precargadas:", *precargar_cuda12(), sep="\n  ")

**Comprobación inmediata.** Se carga el modelo más chico (`colab.modelo_comprobacion`) en `cuda` y se transcribe un segundo de silencio. Eso ya pasa por cuBLAS y cuDNN. Si algo falla con las librerías de CUDA, falla aquí, en segundos, y no en la sección 4.

In [ ]:
import numpy as np
from src.transcripcion import cargar_modelo, liberar_modelos

whisper, carga = cargar_modelo(colab["modelo_comprobacion"], "cuda", cfg_inicial["asr"]["compute_type"]["cuda"])
segmentos, _ = whisper.transcribe(np.zeros(16000, dtype=np.float32), language=cfg_inicial["proyecto"]["idioma"],
                                  vad_filter=False)
list(segmentos)  # la transcripción ocurre al recorrer el generador
print(f"OK: {colab['modelo_comprobacion']} transcribió en device={carga['device']!r}, compute_type={carga['compute_type']!r}")
assert carga["device"] == "cuda"
del whisper
liberar_modelos()

In [ ]:
import sys

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config
from src.metricas import intervalo_bootstrap, wer_cer
from src.normalizacion import normalizar_texto
from src.transcripcion import cargar_modelo, id_corrida, liberar_modelos, transcribir_corpus, vram_usada_mb

cfg = cargar_config()
rutas, asr = cfg["rutas"], cfg["asr"]
SALIDAS = Path(cfg["colab"]["drive_salidas"])
CACHE = SALIDAS / rutas["cache_dir"].name            # se copia a data/cache
RESULTADOS = SALIDAS / rutas["resultados_dir"].name  # se copia a data/resultados
GLOSARIO = [*asr["glosario"]["tecnologias"], *asr["glosario"]["restaurante"]]
ESQUEMA = {"e": "entrevista", "l": "pedido"}
referencias = {clip: (rutas["referencias_dir"] / cfg["archivos"]["patron_referencia"].format(clip=clip))
               .read_text(encoding="utf-8") for clip in cfg["audios"]["archivos"].values()}
pd.set_option("display.max_colwidth", 120)
print(f"{len(referencias)} referencias · salidas en {SALIDAS}")

## 1. Montar Drive y ubicar los audios

Los audios son privados: no están en el repositorio, y el enlace a su carpeta tampoco. Se ubican por su **nombre original** en Drive ("task 1-1" … "task 2-5"), sin renombrarlos. El mapeo nombre → clip está en `config.yaml → audios.archivos`, y desde aquí todo el notebook usa los clips `e1 … l5`.

`src.audios.buscar_audios` recorre `audios.drive_raiz` y todas sus subcarpetas. Acepta cualquier extensión de `audios.extensiones` y no distingue mayúsculas. **Se detiene** si falta alguno de los 10 o si alguno aparece más de una vez, y dice cuáles por su nombre original. En un Drive con muchos archivos, el recorrido puede tardar uno o dos minutos.

In [ ]:
from google.colab import drive

drive.mount(cfg["audios"]["drive_montaje"])

In [ ]:
from src.audios import buscar_audios

audios = buscar_audios(cfg["audios"]["drive_raiz"])
carpetas = sorted({str(ruta.parent) for ruta in audios.values()})
if len(carpetas) == 1:
    print(f"Carpeta encontrada: {carpetas[0]}")
else:
    print("Los audios están en varias carpetas:", *carpetas, sep="\n  ")
for clip, ruta in audios.items():
    print(f"  {clip}  ←  {ruta.name}")

**Confirma que la carpeta de arriba es la de tus audios antes de seguir.** Si aparecen varias carpetas, revisa que no haya una copia antigua con los mismos nombres.

In [ ]:
RESULTADOS.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
corridas = {}  # corrida -> {clip: transcripción}; se llena en cada sección


def medir(resultados):
    """Mediciones por clip de una corrida, contra las referencias."""
    corrida = next(iter(resultados.values()))["corrida"]
    corridas.setdefault(corrida, {}).update(resultados)
    return pd.DataFrame([{
        "corrida": corrida, "clip": clip, "modelo": r["modelo"], "precision": r["compute_type"],
        "device": r["device"], "glosario": r["glosario"] or "sin", **wer_cer(referencias[clip], r["texto"]),
        "duracion_min": round(r["duracion_audio_s"] / 60, 2), "tiempo_s": r["tiempo_s"],
        "s_por_min_audio": r["segundos_por_minuto"], "vram_base_mb": r["vram_base_mb"], "vram_pico_mb": r["vram_pico_mb"],
    } for clip, r in resultados.items()])


def resumen(corrida):
    """Una fila por corrida: WER y CER del corpus, velocidad y VRAM."""
    rs = corridas[corrida]
    clips = list(rs)
    m = wer_cer([referencias[c] for c in clips], [rs[c]["texto"] for c in clips])
    picos = [r["vram_pico_mb"] for r in rs.values() if r["vram_pico_mb"] is not None]
    bases = [r["vram_base_mb"] for r in rs.values() if r["vram_base_mb"] is not None]
    r0 = next(iter(rs.values()))
    return {"corrida": corrida, "modelo": r0["modelo"], "precision": r0["compute_type"],
            "device": "/".join(sorted({r["device"] for r in rs.values()})), "glosario": r0["glosario"] or "sin",
            "clips": len(clips), "wer": round(m["wer"], 4), "cer": round(m["cer"], 4),
            "s_por_min_audio": round(sum(r["tiempo_s"] for r in rs.values())
                                     / (sum(r["duracion_audio_s"] for r in rs.values()) / 60), 2),
            "vram_pico_mb": max(picos) if picos else None,
            "vram_corrida_mb": round(max(picos) - min(bases), 1) if picos and bases else None}

## 2. Duración real de cada audio

Reemplaza la estimación por número de palabras de `01_ground_truth` (columna `en_rango`). Cada audio se decodifica igual que lo hace faster-whisper (16 kHz, mono) y se mide. La tarea pide clips de 2 a 4 minutos (`config.yaml → proyecto.duracion_clip_min`).

In [ ]:
from src.audios import duracion_audio_s

minimo, maximo = cfg["proyecto"]["duracion_clip_min"]
ppm = cfg["proyecto"]["palabras_por_minuto_estimadas"]
duraciones = pd.DataFrame([{
    "clip": clip, "archivo": ruta.name, "duracion_s": (s := duracion_audio_s(ruta)),
    "duracion_min": round(s / 60, 2), "min_estimados": round(len(referencias[clip].split()) / ppm, 1),
} for clip, ruta in audios.items()])
duraciones["en_rango"] = duraciones.duracion_min.between(minimo, maximo)
duraciones.to_csv(RESULTADOS / rutas["duraciones"].name, index=False)
fuera = duraciones[~duraciones.en_rango]
print(f"Total: {duraciones.duracion_min.sum():.1f} min de audio")
print("Todos los clips están entre 2 y 4 minutos." if fuera.empty else
      f"FUERA DE {minimo} A {maximo} MINUTOS: {', '.join(f'{c} ({m} min)' for c, m in zip(fuera["clip"], fuera.duracion_min))}")
duraciones

## 3. Prueba de que se usó la GPU

Un pipeline que cae a CPU en silencio da el mismo texto que uno que usó la GPU; solo la medición los separa. Por eso aquí el device **no sale de `config.yaml`**: se carga un modelo con `device="auto"` y se lee el device que reporta CTranslate2 (`model.model.device`). Si no es `cuda`, el notebook se detiene.

**Cómo se mide la VRAM.** faster-whisper corre sobre CTranslate2, no sobre PyTorch, así que `torch.cuda.max_memory_allocated` no ve su memoria y daría 0. `src.transcripcion` consulta con pynvml la memoria usada de la GPU en un hilo aparte, cada 50 ms (`asr.vram_intervalo_s`), mientras transcribe, y guarda el pico. `vram_base_mb` es la memoria usada antes de cargar el modelo, así que `vram_pico_mb − vram_base_mb` es lo que ocupa esa corrida.

In [ ]:
import ctranslate2

!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.used --format=csv
print("GPUs CUDA visibles para CTranslate2:", ctranslate2.get_cuda_device_count())
print("VRAM usada antes de cargar un modelo:", vram_usada_mb(), "MB")
modelo_prueba, carga = cargar_modelo(asr["cpu"]["modelo"], "auto")
print(f"device='auto' -> CTranslate2 reporta device={carga['device']!r}, compute_type={carga['compute_type']!r}")
print("VRAM usada con el modelo cargado:", vram_usada_mb(), "MB")
assert carga["device"] == "cuda", "La GPU no está activa: Entorno de ejecución → Cambiar tipo → T4 GPU"
del modelo_prueba
liberar_modelos()

## 4. Corrida rápida: un clip de cada tarea

Antes de la corrida completa, `asr.clips_prueba_rapida` (una entrevista y una llamada) se transcriben con el modelo y la precisión del motor, y después se pasan por `procesar()` de punta a punta. Si algo falla (un formato de audio, la GPU, el motor), sale aquí, en dos minutos, y no a la mitad de la corrida completa.

**WER y CER.** Referencia e hipótesis se normalizan con `normalizar_texto` antes de llamar a jiwer: minúsculas, sin puntuación y números a dígitos. Ver `src/normalizacion.py`.

In [ ]:
rapida = {clip: audios[clip] for clip in asr["clips_prueba_rapida"]}
medicion_rapida = medir(transcribir_corpus(rapida, asr["modelo"], "cuda", asr["compute_type"]["cuda"], CACHE))
assert (medicion_rapida.device == "cuda").all(), "alguna transcripción no corrió en la GPU"
medicion_rapida

In [ ]:
from src.engine import procesar

for clip, ruta in rapida.items():
    salida = procesar(str(ruta), ESQUEMA[clip[0]])
    print(f"== {clip} ({ESQUEMA[clip[0]]}) · device {salida['device']} · {salida['tiempo_s']} s · error: {salida['error']}")
    if salida["campos"]:
        display(pd.DataFrame([{"campo": c, "valor": d["valor"], "estado": salida["campos_verificados"][c]}
                              for c, d in salida["campos"].items()]))
    if salida["validacion"]:
        v = salida["validacion"]
        print(f"  validación: esperado {v['esperado']}, dicho {v['dicho']}, cuadra {v['cuadra']} · "
              f"{v['explicacion']['explicacion']} · diagnóstico {v['diagnostico']['diagnostico']}")
    assert salida["error"] is None, salida["error"]

## 5. Modelos y precisiones en GPU

Cada modelo de `asr.modelos_comparacion` (uno pequeño y uno large) se prueba con cada precisión de `asr.precisiones_comparacion` (float16 e int8), sobre los 10 clips y sin glosario. Antes de cada combinación se descargan los modelos anteriores, para que la VRAM de una no se sume a la siguiente.

- `s_por_min_audio`: segundos de cómputo por minuto de audio (menos es más rápido).
- `vram_pico_mb`: el pico de memoria usada en la GPU.
- `vram_corrida_mb`: el pico menos la memoria usada antes de cargar el modelo.

Referencia del enunciado: large a int8 ≈ 2 926 MB contra 4 525 MB a float16.

In [ ]:
por_clip_gpu = []
for modelo in asr["modelos_comparacion"]:
    for precision in asr["precisiones_comparacion"]:
        liberar_modelos()
        print(f"== {modelo} · {precision}")
        por_clip_gpu.append(medir(transcribir_corpus(audios, modelo, "cuda", precision, CACHE)))
por_clip_gpu = pd.concat(por_clip_gpu, ignore_index=True)
comparacion_gpu = pd.DataFrame([resumen(id_corrida(m, p, "cuda", None))
                                for m in asr["modelos_comparacion"] for p in asr["precisiones_comparacion"]])
comparacion_gpu

In [ ]:
por_clip_gpu.pivot_table(index="clip", columns=["modelo", "precision"], values="wer").round(3)

## 6. ¿Cambió el texto entre float16 e int8?

Más rápido y peor no es una mejora. Por eso, para cada modelo y clip, se compara la transcripción en float16 con la de int8:
- `texto_identico`: si las dos son iguales después de normalizar;
- `wer_entre_precisiones`: cuántas palabras cambian, tomando float16 como referencia;
- el WER de cada una contra la referencia humana.

In [ ]:
p0, p1 = asr["precisiones_comparacion"][:2]
filas = []
for modelo in asr["modelos_comparacion"]:
    a, b = corridas[id_corrida(modelo, p0, "cuda", None)], corridas[id_corrida(modelo, p1, "cuda", None)]
    for clip in a:
        filas.append({"modelo": modelo, "clip": clip,
                      "texto_identico": normalizar_texto(a[clip]["texto"]) == normalizar_texto(b[clip]["texto"]),
                      "wer_entre_precisiones": round(wer_cer(a[clip]["texto"], b[clip]["texto"])["wer"], 4),
                      f"wer_{p0}": round(wer_cer(referencias[clip], a[clip]["texto"])["wer"], 4),
                      f"wer_{p1}": round(wer_cer(referencias[clip], b[clip]["texto"])["wer"], 4)})
cambios_precision = pd.DataFrame(filas)
display(cambios_precision.groupby("modelo").agg(clips_identicos=("texto_identico", "sum"), clips=("clip", "count"),
                                                wer_entre_precisiones_medio=("wer_entre_precisiones", "mean")))
cambios_precision

## 7. CPU contra GPU

Los mismos clips, el mismo modelo y la misma precisión en los dos devices. **En CPU se usa solo el modelo pequeño y dos clips** (`asr.cpu`): el large en la CPU de Colab tardaría tanto que la sesión podría expirar. La precisión es int8 en los dos lados, porque float16 no existe en CPU. La corrida de GPU con la que se compara es la de la sección 5.

**Qué precisión reporta la librería.** En CTranslate2, "int8" cuantiza los pesos; el resto del cálculo va en otra precisión, que la librería reporta junto con int8: `int8_float32` en CPU (lo vimos en el ensayo local) y, probablemente, `int8_float16` en GPU. Las columnas `precision` muestran lo que reporta la librería, no lo que se pidió.

In [ ]:
cpu = asr["cpu"]
assert cpu["modelo"] in asr["modelos_comparacion"] and cpu["compute_type"] in asr["precisiones_comparacion"]
liberar_modelos()
en_cpu = transcribir_corpus({c: audios[c] for c in cpu["clips"]}, cpu["modelo"], "cpu", cpu["compute_type"], CACHE)
medicion_cpu = medir(en_cpu)
en_gpu = corridas[id_corrida(cpu["modelo"], cpu["compute_type"], "cuda", None)]
cpu_gpu = pd.DataFrame([{
    "clip": clip, "device_cpu": en_cpu[clip]["device"], "device_gpu": en_gpu[clip]["device"],
    "s_por_min_cpu": en_cpu[clip]["segundos_por_minuto"], "s_por_min_gpu": en_gpu[clip]["segundos_por_minuto"],
    "veces_mas_rapido_gpu": round(en_cpu[clip]["tiempo_s"] / en_gpu[clip]["tiempo_s"], 1),
    "texto_identico": normalizar_texto(en_cpu[clip]["texto"]) == normalizar_texto(en_gpu[clip]["texto"]),
    "wer_cpu": round(wer_cer(referencias[clip], en_cpu[clip]["texto"])["wer"], 4),
    "wer_gpu": round(wer_cer(referencias[clip], en_gpu[clip]["texto"])["wer"], 4),
} for clip in cpu["clips"]])
assert (cpu_gpu.device_cpu == "cpu").all() and (cpu_gpu.device_gpu == "cuda").all()
cpu_gpu

## 8. Glosario

**Qué dice la versión instalada.** La documentación pública es inconsistente sobre este punto, así que se revisó el código de faster-whisper 1.2.1, la versión fijada (`WhisperModel.transcribe` y `get_prompt`). La celda siguiente imprime esa documentación y ese código desde el paquete instalado aquí, para confirmar que es el mismo.

- **`initial_prompt`** se convierte en tokens una sola vez y entra como "texto previo" de la **primera** ventana de 30 s. Después sigue en el contexto solo mientras quepa entre los últimos tokens, y se pierde si hay un reinicio por temperatura. Influye sobre todo al principio del audio.
- **`hotwords`** se antepone al prompt en **cada** ventana, salvo que se pase un `prefix`.
- **Los dos a la vez** están permitidos: `hotwords` va primero y luego el contexto, que empieza con el `initial_prompt`.

**Qué se mide.** El modelo y la precisión del motor, sobre los 10 clips, con cada mecanismo de `asr.glosario_mecanismos_comparacion`, contra la misma corrida sin glosario de la sección 5. El glosario es el vocabulario de dominio de `config.yaml → asr.glosario`. Tiene a propósito términos que no están en el corpus, para que no sea una lista de respuestas. La diferencia de WER se calcula **por clip** (con glosario − sin glosario), y su intervalo de confianza es un bootstrap sobre esas diferencias pareadas.

In [ ]:
import inspect

import faster_whisper
from faster_whisper import WhisperModel

print("faster-whisper", faster_whisper.__version__)
doc = WhisperModel.transcribe.__doc__.splitlines()
for clave in ("initial_prompt:", "hotwords:"):
    i = next(n for n, linea in enumerate(doc) if linea.strip().startswith(clave))
    print("\n".join(doc[i:i + 3]))
print(inspect.getsource(WhisperModel.get_prompt))

In [ ]:
precision = asr["compute_type"]["cuda"]
sin_glosario = id_corrida(asr["modelo"], precision, "cuda", None)
con_glosario = []
for mecanismo in asr["glosario_mecanismos_comparacion"]:
    liberar_modelos()
    print(f"== {asr['modelo']} · {precision} · {mecanismo}")
    medir(transcribir_corpus(audios, asr["modelo"], "cuda", precision, CACHE, GLOSARIO, mecanismo))
    con_glosario.append(id_corrida(asr["modelo"], precision, "cuda", mecanismo))

wer_clip = {c: {clip: wer_cer(referencias[clip], r["texto"])["wer"] for clip, r in corridas[c].items()}
            for c in [sin_glosario, *con_glosario]}
glosario_por_clip = pd.DataFrame(wer_clip).rename(columns={sin_glosario: "sin"} | {c: corridas[c][next(iter(corridas[c]))]["glosario"] for c in con_glosario})
mecanismos = list(glosario_por_clip.columns[1:])
for m in mecanismos:
    glosario_por_clip[f"dif_{m}"] = glosario_por_clip[m] - glosario_por_clip["sin"]
glosario_por_clip.round(4).to_csv(RESULTADOS / rutas["glosario_por_clip"].name, index_label="clip")
glosario_por_clip.round(4)

In [ ]:
def terminos(texto):
    t = f" {normalizar_texto(texto)} "
    return {g for g in GLOSARIO if f" {normalizar_texto(g)} " in t}


filas = []
for m in mecanismos:
    diferencias = list(glosario_por_clip[f"dif_{m}"])
    bajo, alto = intervalo_bootstrap(diferencias)
    filas.append({"mecanismo": m, "dif_wer_media": round(sum(diferencias) / len(diferencias), 4),
                  "ic95_bootstrap_pareado": (round(bajo, 4), round(alto, 4)),
                  "clips_mejoran": sum(d < 0 for d in diferencias), "clips_empeoran": sum(d > 0 for d in diferencias)})
for corrida, nombre in [(sin_glosario, "sin"), *zip(con_glosario, mecanismos)]:
    esperados = sum(len(terminos(referencias[c])) for c in corridas[corrida])
    acertados = sum(len(terminos(referencias[c]) & terminos(r["texto"])) for c, r in corridas[corrida].items())
    print(f"{nombre:>15}: términos del glosario dichos en las referencias y bien transcritos: {acertados} de {esperados}")
pd.DataFrame(filas)

Una diferencia negativa significa que el glosario baja el WER. Si el intervalo incluye el 0, con 10 clips no se puede afirmar que el glosario ayude ni que perjudique. El conteo de términos bien transcritos mira justo lo que el glosario intenta mejorar (los nombres propios), que el WER del texto completo diluye.

## 9. Caché

Cada transcripción quedó como JSON en `HW_05_salidas/cache/transcripciones/<corrida>/<clip>.json`, en Drive. Cada uno guarda el texto, las palabras con su probabilidad, los segmentos, el modelo, la precisión y el device que reportó la librería, el tiempo, la VRAM y las versiones. Al terminar, esta carpeta `cache/` se copia a `data/cache/` del repo y se commitea: la app y cualquier workflow de CI leen de ahí y nunca vuelven a transcribir.

In [ ]:
import json

archivos = sorted(CACHE.rglob("*.json"))
display(pd.Series([a.parent.name for a in archivos]).value_counts().rename("clips").to_frame())
ejemplo = json.loads(archivos[0].read_text(encoding="utf-8"))
print(archivos[0].relative_to(SALIDAS))
print({k: (v if k not in ("texto", "palabras", "segmentos") else f"<{len(v)} elementos>" if isinstance(v, list) else v[:80] + "…")
       for k, v in ejemplo.items()})

## 10. Resumen de todas las mediciones

Una fila por corrida. Se guarda en `resultados/` junto con el WER por clip.

In [ ]:
resumen_total = pd.DataFrame([resumen(c) for c in corridas]).sort_values(["device", "modelo", "precision", "glosario"])
resumen_total.to_csv(RESULTADOS / rutas["mediciones_transcripcion"].name, index=False)
pd.concat([medir(corridas[c]) for c in list(corridas)], ignore_index=True).round(4).to_csv(
    RESULTADOS / rutas["wer_por_clip"].name, index=False)
print("Guardado en", RESULTADOS, ":", sorted(p.name for p in RESULTADOS.glob("*.csv")))
resumen_total